In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/districts_master.csv')
print(df.shape)
df.head()

(736, 28)


,state,district,population,households,literacy_rate_pct,sc_population,st_population,sub_centres,phcs,chcs,...,drinking_water_pct,electricity_pct,health_insurance_pct,women_anaemia_pct,sub_centres_per_lakh,phcs_per_lakh,chcs_per_lakh,district_hospitals_per_lakh,sc_population_pct,st_population_pct
0,Andaman And Nicobar Islands,Nicobars,36842.0,9288.0,68.758482,0.0,23681.0,41.0,4.0,1.0,...,98.8,97.9,2.7,38.3,111.286032,10.857174,2.714293,2.714293,0.000000,64.277184
1,Andaman And Nicobar Islands,North And Middle Andaman,105597.0,26199.0,74.512534,0.0,758.0,44.0,8.0,2.0,...,92.2,93.2,2.1,62.1,41.667850,7.575973,1.893993,0.946997,0.000000,0.717823
2,Andaman And Nicobar Islands,South Andamans,238142.0,59064.0,79.896028,0.0,4091.0,39.0,15.0,1.0,...,97.9,99.6,1.2,57.7,16.376784,6.298763,0.419918,0.419918,0.000000,1.717883
3,Andhra Pradesh,Anantapur,4081148.0,968160.0,56.625244,583135.0,154127.0,586.0,108.0,15.0,...,98.8,99.6,73.2,50.5,14.358705,2.646314,0.367544,0.024503,14.288504,3.776560
4,Andhra Pradesh,Chittoor,4174064.0,1039953.0,63.915599,785760.0,159165.0,644.0,120.0,15.0,...,98.5,99.7,70.7,51.8,15.428609,2.874896,0.359362,0.047915,18.824819,3.813190


In [2]:
# Indicators where HIGHER value = BETTER healthcare access
positive_indicators = [
    'sub_centres_per_lakh', 'phcs_per_lakh', 'chcs_per_lakh', 'district_hospitals_per_lakh',
    'institutional_delivery_pct_hmis', 'csection_rate_pct', 'institutional_births_pct',
    'skilled_birth_attendance_pct', 'child_immunization_pct',
    'sanitation_pct', 'drinking_water_pct', 'electricity_pct', 'health_insurance_pct',
    'literacy_rate_pct',
]

# Indicators where HIGHER value = WORSE healthcare access (need to flip)
negative_indicators = [
    'women_anaemia_pct',
]

print("Positive indicators:", len(positive_indicators))
print("Negative indicators:", len(negative_indicators))

Positive indicators: 14
Negative indicators: 1


In [4]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Scale positive indicators (0 = worst, 1 = best)
df_scaled_pos = pd.DataFrame(
    scaler.fit_transform(df[positive_indicators]),
    columns=[f"{col}_scaled" for col in positive_indicators],
    index=df.index
)

# Scale negative indicators, then FLIP so 1 = best (low anaemia)
df_scaled_neg = pd.DataFrame(
    scaler.fit_transform(df[negative_indicators]),
    columns=[f"{col}_scaled" for col in negative_indicators],
    index=df.index
)
df_scaled_neg = 1 - df_scaled_neg  # flip direction

df = pd.concat([df, df_scaled_pos, df_scaled_neg], axis=1)
print(df.shape)
df[[c for c in df.columns if c.endswith('_scaled')]].describe()

(736, 43)


,sub_centres_per_lakh_scaled,phcs_per_lakh_scaled,chcs_per_lakh_scaled,district_hospitals_per_lakh_scaled,institutional_delivery_pct_hmis_scaled,csection_rate_pct_scaled,institutional_births_pct_scaled,skilled_birth_attendance_pct_scaled,child_immunization_pct_scaled,sanitation_pct_scaled,drinking_water_pct_scaled,electricity_pct_scaled,health_insurance_pct_scaled,literacy_rate_pct_scaled,women_anaemia_pct_scaled
count,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000,736.000000
mean,0.151994,0.073134,0.042766,0.031246,0.911111,0.275660,0.854068,0.847680,0.769317,0.607902,0.891525,0.907035,0.404166,0.556402,0.482117
std,0.103587,0.073181,0.071243,0.079997,0.144904,0.231043,0.152998,0.143502,0.141231,0.199761,0.150787,0.135354,0.237837,0.175101,0.155612
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.097722,0.038775,0.019453,0.004298,0.890083,0.074766,0.788486,0.777135,0.692750,0.469590,0.863095,0.889241,0.199534,0.435309,0.382634
50%,0.126404,0.054816,0.028323,0.010651,0.976422,0.218959,0.900127,0.885673,0.777000,0.637907,0.948980,0.957278,0.357143,0.550148,0.464377
75%,0.181909,0.077838,0.042616,0.019590,0.997226,0.450267,0.966921,0.955137,0.872250,0.765559,0.986395,0.984177,0.608954,0.692351,0.567430
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
scaled_cols = [c for c in df.columns if c.endswith('_scaled')]

df['healthcare_access_score'] = df[scaled_cols].mean(axis=1)

print(df['healthcare_access_score'].describe())

count    736.000000
mean       0.520408
std        0.069448
min        0.328621
25%        0.472327
50%        0.530163
75%        0.573582
max        0.674578
Name: healthcare_access_score, dtype: float64


In [6]:
# Split into 4 tiers using quartiles
df['coverage_tier'] = pd.qcut(
    df['healthcare_access_score'],
    q=4,
    labels=['Critical', 'Underserved', 'Adequate', 'Well Covered']
)

print(df['coverage_tier'].value_counts())

coverage_tier
Critical        184
Underserved     184
Adequate        184
Well Covered    184
Name: count, dtype: int64


In [7]:
print("TOP 5 districts (best access):")
print(df.nlargest(5, 'healthcare_access_score')[['state', 'district', 'healthcare_access_score', 'coverage_tier']])

print("\nBOTTOM 5 districts (worst access):")
print(df.nsmallest(5, 'healthcare_access_score')[['state', 'district', 'healthcare_access_score', 'coverage_tier']])

TOP 5 districts (best access):
          state       district  healthcare_access_score coverage_tier
480  Puducherry           Mahe                 0.674578  Well Covered
553  Tamil Nadu  Kanniyakumari                 0.671985  Well Covered
295      Kerala      Alappuzha                 0.661409  Well Covered
300      Kerala         Kollam                 0.660413  Well Covered
297      Kerala         Idukki                 0.659827  Well Covered

BOTTOM 5 districts (worst access):
             state            district  healthcare_access_score coverage_tier
424      Meghalaya  West Jaintia Hills                 0.328621      Critical
634  Uttar Pradesh            Bahraich                 0.334286      Critical
636  Uttar Pradesh           Balrampur                 0.338032      Critical
90           Bihar          Kishanganj                 0.350051      Critical
74           Bihar              Araria                 0.352875      Critical


In [8]:
df.to_csv('../data/processed/districts_with_tiers.csv', index=False)
print("Saved:", df.shape)

Saved: (736, 45)
